# Attention U-Net → BUSI — project notebook

Orchestration for Google Colab: **bootstrap → data → experiments → figures**.
All logic lives in the `busi/` package; this notebook only orchestrates.
The control panel is `busi/config.py` — switch model/loss/params there or per run below.

See `../PLAN.md` for the milestone experiment matrix (Core → CBAM → scSE).

## 1. Bootstrap (Colab)

The repo is **private**, so cloning on Colab needs auth. Two options:
**A.** clone with a GitHub token stored in Colab *Secrets* (key `GH_TOKEN`); or
**B.** keep the repo in Google Drive and `cd` into it. Running locally? skip this cell.

In [ ]:
import os, sys, subprocess

IN_COLAB = "google.colab" in sys.modules
REPO_DIR = "attention-unet-busi"

if IN_COLAB:
    from google.colab import drive
    drive.mount("/content/drive")
    try:
        from google.colab import userdata
        token = userdata.get("GH_TOKEN")
        url = f"https://{token}@github.com/YuvalDziesietnik/attention-unet-busi.git"
        if not os.path.isdir(REPO_DIR):
            subprocess.run(["git", "clone", url, REPO_DIR], check=True)
        os.chdir(REPO_DIR)
    except Exception as e:
        print("Token clone skipped/failed:", e)
        print('Fall back: os.chdir("/content/drive/MyDrive/attention-unet-busi")')
    subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                    "-r", "requirements.txt", "-i", "https://pypi.org/simple"], check=True)

print("cwd:", os.getcwd())

## 2. Sanity check

In [ ]:
import torch, busi
from busi.config import Config
from busi import train as T
print("busi", busi.__version__, "| torch", torch.__version__, "| device", T.get_device())

## 3. Data — download & verify

Download BUSI from Kaggle (`aryashah2k/breast-ultrasound-images-dataset`) and put the
`benign/ malignant/ normal/` folders under `Config.data_root` (default `data/BUSI`).
On Colab, point `data_root` at your Drive copy. The train/val/test split is created
once (fixed seed) and persisted to `splits/split.json` — shared by every run.

In [ ]:
from busi import experiment as E
from busi.data import list_busi_samples

cfg = Config()                       # <-- control panel
samples = list_busi_samples(cfg.data_root, classes=tuple(cfg.classes))
print("total samples:", len(samples))
split = E.get_or_make_split(cfg)
print({k: len(v) for k, v in split.items()})

## 4. Core experiments (baseline vs attention)

Trains the plain U-Net (C1) and the Attention U-Net (C2), each over 3 seeds, and
aggregates mean±std. This is the heavy step — **run on a Colab GPU**. Set `QUICK=True`
for a fast local smoke (1 seed, few epochs).

In [ ]:
QUICK = False    # True -> 1 seed + few epochs, for a fast local smoke
base = Config(epochs=(3 if QUICK else 100), seeds=([42] if QUICK else [42, 1, 7]))

results = []
for name in ["unet", "attention_unet"]:   # Core milestone; add cbam_unet/scse_unet later
    results.append(E.run_seeds(name, cfg=base, seeds=base.seeds))

## 5. Results table & qualitative figures

In [ ]:
print(E.make_results_table(results))

# Attention-map figures for the attention model (best-seed checkpoint).
att = next(r for r in results if r["model"] == "attention_unet")
cfg_att = E._cfg_for(base, "attention_unet", att["seeds"][0])
ckpt = f"{cfg_att.checkpoints_dir}/{cfg_att.experiment_name}.pt"
figs = E.save_prediction_figures(cfg_att, ckpt, "results/figures", n=6)
print("saved figures:", figs)

## 6. Later milestones

- **Desired (Stage 7):** add `"cbam_unet"` to the loop above.
- **Stretch (Stage 8):** add `"scse_unet"`, and a Focal-Tversky run
  (`Config(loss_name="focal_tversky")`) on the best variant.

Results JSON land in `results/`, figures in `results/figures/`, checkpoints in
`checkpoints/` (on Colab, keep these on Drive so a disconnect doesn't lose them).